# 02 - Preprocessing & Feature Engineering
## Building Energy Efficiency: Heating & Cooling Load Prediction
**DEPI Graduation Project - Team CloudXAi (NHA-5-165)**

This notebook covers:
1. Train/Test splitting without data leakage (80/20 stratified random seed).
2. Domain-driven feature engineering: `Wall_Surface_Ratio` and `Glazing_x_Height`.
3. Preprocessor pipeline construction with `StandardScaler`, `OneHotEncoder`, and `FunctionTransformer`.


In [ ]:
import os
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LinearRegression

from src.energy_efficiency.config import FEATURES, TARGETS, RANDOM_STATE, TEST_SIZE
from src.energy_efficiency.data_loader import load_raw_data, get_train_test_split
from src.energy_efficiency.feature_engineering import add_features
from src.energy_efficiency.preprocessing import build_preprocessor

print("Preprocessing modules loaded.")


### 1. Train / Test Split


In [ ]:
df = load_raw_data()
X_train, X_test, y_train, y_test = get_train_test_split(df)
print(f"Train Shape: X={X_train.shape}, y={y_train.shape}")
print(f"Test Shape:  X={X_test.shape}, y={y_test.shape}")


### 2. Feature Engineering Logic
We create two physical interaction terms:
- $\text{Wall\_Surface\_Ratio} = \frac{\text{Wall\_Area}}{\text{Surface\_Area}}$
- $\text{Glazing\_x\_Height} = \text{Glazing\_Area} \times \text{Overall\_Height}$


In [ ]:
engineered_train = add_features(X_train)
engineered_train[["Wall_Area", "Surface_Area", "Wall_Surface_Ratio", "Glazing_Area", "Overall_Height", "Glazing_x_Height"]].head()


### 3. Feature Engineering Cross-Validation Experiments


In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
configs = [
    ("Baseline (original features)", False, ()),
    ("New features (Wall_Surface_Ratio, Glazing_x_Height)", True, ()),
    ("New features + drop Surface_Area", True, ("Surface_Area",)),
    ("Drop Surface_Area only", False, ("Surface_Area",)),
    ("Drop Surface_Area + Roof_Area", False, ("Surface_Area", "Roof_Area")),
]

exp_results = []
for name, eng, drop in configs:
    pipe = build_preprocessor(scale=True, onehot=True, engineer=eng, drop=drop)
    X_tr = pipe.fit_transform(X_train)
    lr = LinearRegression()
    r2_scores = cross_val_score(lr, X_tr, y_train["Heating_Load"], cv=kf, scoring="r2")
    exp_results.append({
        "Experiment": name,
        "CV R² Mean": round(r2_scores.mean(), 4),
        "CV R² Std": round(r2_scores.std(), 4),
    })

pd.DataFrame(exp_results).sort_values("CV R² Mean", ascending=False)


### 4. Build and Save Preprocessor Pipelines


In [ ]:
prep_linear = build_preprocessor(scale=True, onehot=True, engineer=True)
prep_tree = build_preprocessor(scale=False, onehot=False, engineer=True)

prep_linear.fit(X_train)
prep_tree.fit(X_train)

print("Linear Transformed Dimension:", prep_linear.transform(X_train).shape)
print("Tree Transformed Dimension:", prep_tree.transform(X_train).shape)


### 5. Export Processed Datasets


In [ ]:
os.makedirs("data/processed", exist_ok=True)
pd.concat([X_train, y_train], axis=1).to_csv("data/processed/train.csv", index=False)
pd.concat([X_test, y_test], axis=1).to_csv("data/processed/test.csv", index=False)
print("Processed data saved to data/processed/")
